> **Kaggle kernel `dr-s5-train`.** The 4 training runs (soft labels, sigma 0.5 x 3 seeds, sigma 1.0 x 1 seed) and the analysis. Built by `experiments/07_soft_labels/kaggle/build_kernels.py` from `experiments/07_soft_labels/07_soft_labels.ipynb`; only the Drive setup and the QUICK_TEST switch differ.

# Stage 5: Soft Ordinal Labels for Diabetic Retinopathy Grading
### Can Gaussian soft-label targets give the benefit of ordinal regression while keeping five class probabilities?

**Hypothesis (M3, written before this notebook was run):** [`experiments/07_soft_labels/hypothesis.md`](hypothesis.md). Replacing the one-hot cross-entropy target with a **Gaussian soft-label** target over the grades (sigma = 0.5) raises QWK and reduces large grading errors, because the loss then knows that Moderate is closer to Severe than to No DR, while the network keeps five outputs and does not shrink the rare grades towards the middle.

**Baselines (not retrained).** Arms A (cross-entropy, vote), B (cross-entropy + tuned cut-offs) and D (ordinal MSE + tuned cut-offs) come from Stage 4 and are read from `experiments/06_imbalance_losses/results/results_per_seed.csv` in the project repository. The seed-42 Kaggle control runs (`ce_s42_k`, `mse_s42_k`) give the paired bootstrap, because they use the same split and the same 550 test images.

| Arm | Trained with | Output -> grade | Question |
|---|---|---|---|
| **G** | soft-label CE (sigma 0.5) | 10-snapshot majority vote | does the soft target help under the paper's rule? (compare A) |
| **H** | soft-label CE (same model as G) | expected grade + validation-tuned cut-offs | **the hypothesis** (compare B and D) |

Runs: `soft_s42`, `soft_s43`, `soft_s44` (sigma 0.5) and `soft_sigma1_s42` (sigma 1.0, ablation). About 30 minutes each on a Colab T4.

**How to run (Colab)**

1. **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**, and allow Google Drive access.
2. The image cache from the Stage 3 notebook (`MyDrive/dr-grading/aptos512.zip`) is reused if it exists, otherwise APTOS is downloaded once.
3. Stopped or disconnected? Run all again. Finished runs are loaded from Drive and the unfinished one resumes after its last epoch.
4. Set `QUICK_TEST = True` in section 1 first to check the whole pipeline in a few minutes.

## 1. Setup and settings

Every hyper-parameter lives in one `Config`. The defaults are the Stage 3 baseline; an experiment changes only `loss` (`"ce"` or `"mse"`), `seed` and `dedupe`.

In [ ]:
import hashlib
import io
import json
import math
import os
import random
import subprocess
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, fields, replace
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

try:
    import timm
except ImportError:                      # not preinstalled on this machine
    !pip install -q timm
    import timm

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "PDR"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Kaggle: results go to /kaggle/working (this kernel's output); images are downloaded to the local disk.
WORK_DIR = Path("/kaggle/working/dr-grading")
LOCAL_DIR = Path("/tmp/dr")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU found (very slow!)")

In [ ]:
QUICK_TEST = False        # True = tiny run (few epochs, few batches, separate output folder) to check the pipeline
CHECKPOINT_EVERY = 1      # save a resumable checkpoint every N epochs
SPLIT_SEED = 42           # the train/val/test split is the same for every run (same as Stage 3)


@dataclass
class Config:
    # The Stage 3 baseline `2_aug_lr1e-4`: paper settings plus augmentation and lr 1e-4.
    image_size: int = 300
    epochs: int = 60
    batch_size: int = 16
    lr: float = 1e-4
    weight_decay: float = 1e-4
    grad_clip: float = 0.1
    n_snapshots: int = 10
    seed: int = 42                # training seed (weights init, batch order, augmentation); the split stays fixed
    backbone: str = "efficientnet_b3"
    pretrained: bool = True
    use_amp: bool = True
    augment: bool = True
    loss: str = "soft"            # "ce" = one-hot cross-entropy (Stage 3); "mse" = ordinal regression (Stage 4); "soft" = Gaussian soft labels (Stage 5)
    sigma: float = 0.5            # width of the Gaussian soft label, in grades (only for loss="soft")
    dedupe: bool = False          # True = drop conflicting duplicates and keep one copy of each identical image
    max_train_batches: int = 0    # 0 = all batches; >0 only for the quick test


BASE_CFG = Config()
SEEDS = [42, 43, 44]
STAGE5_DIR = WORK_DIR / "stage5"
if QUICK_TEST:
    BASE_CFG = replace(BASE_CFG, epochs=4, n_snapshots=2, max_train_batches=5, image_size=224)
    SEEDS = [42, 43]
    STAGE5_DIR = WORK_DIR / "stage5_quick_test"            # never mixed with the real runs
if device.type != "cuda" and not QUICK_TEST:
    raise RuntimeError("This runtime has no GPU (Colab falls back to CPU when the free GPU time runs out). "
                       "Runtime -> Change runtime type -> T4 GPU, then Run all again. On a CPU the settings would "
                       "not match the saved runs, and training would take days.")
if device.type != "cuda":
    BASE_CFG = replace(BASE_CFG, use_amp=False)             # mixed precision is a GPU feature
RUNS_DIR, FIG_DIR = STAGE5_DIR / "runs", STAGE5_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {f"soft_s{seed}": replace(BASE_CFG, loss="soft", sigma=0.5, seed=seed) for seed in SEEDS}
EXPERIMENTS[f"soft_sigma1_s{SEEDS[0]}"] = replace(BASE_CFG, loss="soft", sigma=1.0, seed=SEEDS[0])      # ablation (Q7)

print(json.dumps(asdict(BASE_CFG), indent=2))
print("experiments:", ", ".join(EXPERIMENTS))

## 2. Data

APTOS 2019: 3,662 labelled fundus photographs, five grades. Same source and cache as Stage 3: the full-resolution Hugging Face mirror [`sngsfydy/aptos`](https://huggingface.co/datasets/sngsfydy/aptos), shrunk to 512 px on the long side, with a zipped copy on Drive. Because the image order comes from the same `labels.csv`, the split below is **identical to Stage 3**.

In [ ]:
HF_REPO = "sngsfydy/aptos"
IMG_DIR = LOCAL_DIR / "aptos512"
DRIVE_CACHE = WORK_DIR / "aptos512.zip"
MAX_SIDE = 512


def list_shards():
    with urllib.request.urlopen(f"https://huggingface.co/api/datasets/{HF_REPO}/tree/main/data") as r:
        return sorted(f["path"] for f in json.load(r) if f["path"].endswith(".parquet"))


def convert_shard(parquet_path, out_dir):
    # Save every image in one parquet shard as a <hash>.jpg (<= 512 px). Returns [(file name, grade), ...].
    rows = []
    pf = pq.ParquetFile(parquet_path)
    for group in range(pf.num_row_groups):
        table = pf.read_row_group(group).to_pydict()
        for img, label in zip(table["image"], table["label"]):
            raw = img["bytes"]
            name = hashlib.sha1(raw).hexdigest()[:16] + ".jpg"     # identical photos get identical names
            im = Image.open(io.BytesIO(raw)).convert("RGB")
            scale = MAX_SIDE / max(im.size)
            im.resize((round(im.width * scale), round(im.height * scale)), Image.LANCZOS).save(out_dir / name, quality=95)
            rows.append((name, int(label)))
    return rows


def download_aptos(out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for shard in list_shards():
        local = LOCAL_DIR / Path(shard).name
        print("downloading", local.name, flush=True)
        subprocess.run(["curl", "-sSL", "--retry", "5", "-o", str(local),
                        f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/{shard}"], check=True)
        rows += convert_shard(local, out_dir)
        local.unlink()                                             # free the disk before the next shard
        print(f"  {len(rows)} images so far", flush=True)
    pd.DataFrame(rows, columns=["id_code", "label"]).to_csv(out_dir / "labels.csv", index=False)


if not (IMG_DIR / "labels.csv").exists():
    if DRIVE_CACHE.exists():                                       # Stage 3 (or an earlier session) did the download
        print("restoring cached images from Drive ...")
        subprocess.run(["unzip", "-q", "-o", str(DRIVE_CACHE), "-d", str(IMG_DIR)], check=True)
    else:
        download_aptos(IMG_DIR)
        print("saving a cache to Drive for next time ...")
        pass

df = pd.read_csv(IMG_DIR / "labels.csv")
df["path"] = df["id_code"].map(lambda name: IMG_DIR / name)
labels = df["label"].to_numpy()
print(f"{len(df)} images, {df['id_code'].nunique()} distinct files")

### Duplicates and the two splits

APTOS contains byte-identical images, some with **conflicting grades**. The main experiments use the Stage 3 split (all images, stratified 70/15/15, seed 42). The de-duplication check (prediction P8) uses a second split built after **removing every image whose copies disagree** and **keeping one copy of each remaining identical group**, so no test image has a twin in training.

In [ ]:
def file_hash(path):
    return hashlib.md5(path.read_bytes()).hexdigest()


with ThreadPoolExecutor(os.cpu_count()) as pool:
    df["hash"] = list(tqdm(pool.map(file_hash, df["path"]), total=len(df), desc="hashing"))

groups = df.groupby("hash")["label"].agg(["size", "nunique"])
conflict_hashes = set(groups.index[groups["nunique"] > 1])
df["conflicting_duplicate"] = df["hash"].isin(conflict_hashes)
print(f"{int((groups['size'] > 1).sum())} groups of identical images, {len(conflict_hashes)} of them with conflicting grades")


def make_split(positions, seed=SPLIT_SEED, val_frac=0.15, test_frac=0.15):
    # Stratified train/val/test row positions (into df). With all positions this is exactly the Stage 3 split.
    train_val, test = train_test_split(positions, test_size=test_frac, stratify=labels[positions], random_state=seed)
    train, val = train_test_split(train_val, test_size=val_frac / (1 - test_frac),
                                  stratify=labels[train_val], random_state=seed)
    return {"train": np.sort(train), "val": np.sort(val), "test": np.sort(test)}


keep = ~df["conflicting_duplicate"] & ~df.duplicated("hash")      # de-duplicated data set
SPLITS = {False: make_split(np.arange(len(df))), True: make_split(np.flatnonzero(keep.to_numpy()))}

for dedupe, split in SPLITS.items():
    train_hashes = set(df["hash"].to_numpy()[split["train"]])
    leaked = sum(h in train_hashes for h in df["hash"].to_numpy()[split["test"]])
    print(f"\n{'de-duplicated' if dedupe else 'all images (Stage 3 split)'}: {sum(map(len, split.values()))} images, "
          f"{leaked} test images with a twin in training")
    print(pd.DataFrame({name: np.bincount(labels[idx], minlength=NUM_CLASSES) for name, idx in split.items()},
                       index=CLASS_NAMES).T.to_string())

## 3. Pre-processing (unchanged from Stage 3)

CLAHE on the LAB lightness channel (clip 2.0, 8x8 tiles), 5x5 Gaussian blur, square resize to 300 px. Computed once for all images and kept in memory (~1 GB).

In [ ]:
def preprocess(path, size):
    # Load -> shrink to <= 512 px -> CLAHE + blur -> square resize. Returns RGB uint8 (size, size, 3).
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    h, w = bgr.shape[:2]
    if max(h, w) > 512:
        scale = 512 / max(h, w)
        bgr = cv2.resize(bgr, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    bgr = cv2.GaussianBlur(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), (5, 5), 0)
    bgr = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


paths = df["path"].tolist()
images = np.empty((len(paths), BASE_CFG.image_size, BASE_CFG.image_size, 3), dtype=np.uint8)


def fill(i):
    images[i] = preprocess(paths[i], BASE_CFG.image_size)


t0 = time.time()
with ThreadPoolExecutor(os.cpu_count()) as pool:            # OpenCV releases the GIL, so threads are enough
    list(tqdm(pool.map(fill, range(len(paths))), total=len(paths), desc="pre-processing"))
print(f"{images.shape}, {images.nbytes / 1e9:.2f} GB, {time.time() - t0:.0f}s")

## 4. Model: the only architectural difference is the last layer

EfficientNet-B3 (ImageNet weights) with the paper's head. The baseline ends in **5 outputs** (one score per grade, softmax + cross-entropy). The ordinal model ends in **1 output**: a number on the 0-4 scale, trained with mean-squared error against the true grade. Everything before the last layer is identical.

```
1536 -> Linear 512 -> Dropout 0.5 -> ReLU -> Linear 512 -> Dropout 0.25 -> ReLU -> Linear 5   (baseline, "ce")
                                                                               -> Linear 1   (ordinal,  "mse")
```

**Why MSE:** an error of *d* grades costs *d*², which is exactly the quadratic weight QWK uses. Cross-entropy charges every wrong grade the same.

In [ ]:
def n_outputs(cfg):
    return 1 if cfg.loss == "mse" else NUM_CLASSES


class DRNet(nn.Module):
    def __init__(self, backbone="efficientnet_b3", pretrained=True, outputs=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool="avg")
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, 512), nn.Dropout(0.5), nn.ReLU(inplace=True),
            nn.Linear(512, 512), nn.Dropout(0.25), nn.ReLU(inplace=True),
            nn.Linear(512, outputs),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


_mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
_std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)
_black = -_mean / _std     # what a black pixel looks like after normalisation


def to_model_input(batch_uint8):
    # numpy (B, H, W, 3) uint8 -> normalised float tensor (B, 3, H, W) on the GPU.
    x = torch.from_numpy(batch_uint8).to(device).permute(0, 3, 1, 2).float() / 255.0
    return ((x - _mean) / _std).contiguous(memory_format=torch.channels_last)


def augment_batch(x):
    # Random horizontal flip + random rotation (0-360 degrees), as in Stage 3. Empty corners stay black.
    n = x.shape[0]
    x = x - _black
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)
    angle = torch.rand(n, device=x.device) * 2 * math.pi
    cos, sin, zero = angle.cos(), angle.sin(), torch.zeros_like(angle)
    theta = torch.stack([torch.stack([cos, -sin, zero], 1), torch.stack([sin, cos, zero], 1)], 1)    # (n, 2, 3)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    return (x + _black).contiguous(memory_format=torch.channels_last)

## 5. From model output to a grade, and the metrics

A model's raw output for an image is either 5 probabilities (`ce`) or one number (`mse`). Both can be turned into a **continuous severity score**: the expected grade Σ k·p(k) for `ce`, the number itself for `mse`. A score becomes a grade through four **cut-offs**: rounding uses 0.5 / 1.5 / 2.5 / 3.5; the tuned rule searches each cut-off on a 0.01 grid to maximise **validation** QWK (coordinate search, three passes). The test set never influences a cut-off.

**Metrics** (all on the test set): QWK, accuracy, macro-F1, recall per grade, errors of two or more grades, the confusion matrix, and the screening view **referable DR = grade >= 2**: sensitivity, specificity and AUC. For a fair AUC across both models, the score for "grade >= k" is Σ_{j>=k} p(j) for `ce` and the regression output for `mse`; *ordinal AUC* is the mean over k = 1..4.

In [ ]:
ROUNDING = np.array([0.5, 1.5, 2.5, 3.5])


def qwk(y, pred, k=NUM_CLASSES):
    # Quadratic weighted kappa from the confusion matrix (same value as sklearn's cohen_kappa_score, much faster).
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k).astype(float)
    weights = np.subtract.outer(np.arange(k), np.arange(k)) ** 2 / (k - 1) ** 2
    expected = np.outer(cm.sum(1), cm.sum(0)) / cm.sum()
    denom = (weights * expected).sum()
    return float(1 - (weights * cm).sum() / denom) if denom > 0 else 0.0


def severity_score(out):
    # (..., 5) probabilities -> expected grade; (..., 1) regression output -> that number.
    return out @ np.arange(NUM_CLASSES) if out.shape[-1] == NUM_CLASSES else out[..., 0]


def at_least_score(out, k):
    # Score for "grade >= k": P(grade >= k) for ce, the regression output for mse (any monotone score works for AUC).
    return out[..., k:].sum(-1) if out.shape[-1] == NUM_CLASSES else out[..., 0]


def apply_cutoffs(score, cutoffs):
    return np.digitize(score, cutoffs).astype(int)


def fit_cutoffs(score, y, step=0.01, passes=3):
    # Coordinate search: move one cut-off at a time to the grid value with the best QWK, keeping them ordered.
    cutoffs = ROUNDING.copy()
    best = qwk(y, apply_cutoffs(score, cutoffs))
    for _ in range(passes):
        for i in range(len(cutoffs)):
            lo = cutoffs[i - 1] + step if i > 0 else -0.5
            hi = cutoffs[i + 1] - step if i < len(cutoffs) - 1 else NUM_CLASSES - 0.5
            for value in np.arange(lo, hi + 1e-9, step):
                trial = cutoffs.copy()
                trial[i] = value
                score_trial = qwk(y, apply_cutoffs(score, trial))
                if score_trial > best:                       # strictly better: ties keep the current cut-off
                    best, cutoffs = score_trial, trial
    return cutoffs


def majority_vote(snapshot_probs):
    # snapshot_probs: (n_snapshots, n_images, 5) -> one predicted grade per image (paper's ensemble rule).
    votes = snapshot_probs.argmax(-1)
    tally = np.stack([(votes == c).sum(0) for c in range(NUM_CLASSES)], 1).astype(float)
    tally += 1e-3 * snapshot_probs.mean(0)                   # tie-break
    return tally.argmax(1)


def safe_auc(y_binary, score):
    return float(roc_auc_score(y_binary, score)) if 0 < y_binary.sum() < len(y_binary) else float("nan")


def compute_metrics(y, pred, out):
    _, rec, _, _ = precision_recall_fscore_support(y, pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    ref_true, ref_pred = y >= 2, pred >= 2
    return {
        "qwk": qwk(y, pred),
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", labels=list(range(NUM_CLASSES)), zero_division=0)),
        **{f"recall_{name}": float(rec[c]) for c, name in enumerate(CLASS_NAMES)},
        "off_by_2_or_more": int(np.sum(np.abs(y - pred) >= 2)),
        "ref_sensitivity": float((ref_pred & ref_true).sum() / max(ref_true.sum(), 1)),
        "ref_specificity": float((~ref_pred & ~ref_true).sum() / max((~ref_true).sum(), 1)),
        "ref_auc": safe_auc(ref_true, at_least_score(out, 2)),
        "ordinal_auc": float(np.nanmean([safe_auc(y >= k, at_least_score(out, k)) for k in range(1, NUM_CLASSES)])),
        "confusion_matrix": confusion_matrix(y, pred, labels=list(range(NUM_CLASSES))).tolist(),
    }

In [ ]:
ARMS = {"ce": ("A", "B"), "mse": ("C", "D"), "soft": ("G", "H")}     # (default rule, tuned-cut-off rule) for each trained model


def decide(run, mode):
    # Grades for the test set from one finished run, for both of its arms.
    # mode "single" = the best-validation epoch; "ensemble" = the 10 snapshots.
    if mode == "single":
        val_out, test_out = run["best_val"], run["best_test"]
    else:
        val_out, test_out = run["snap_val"].mean(0), run["snap_test"].mean(0)
    default_arm, tuned_arm = ARMS[run["config"].loss]
    if run["config"].loss in ("ce", "soft"):       # A / G: arg-max, or the paper's majority vote
        default = test_out.argmax(1) if mode == "single" else majority_vote(run["snap_test"])
    else:                                           # C: round the regression output
        default = apply_cutoffs(severity_score(test_out), ROUNDING)
    cutoffs = fit_cutoffs(severity_score(val_out), run["y_val"])          # B / D: cut-offs from validation only
    tuned = apply_cutoffs(severity_score(test_out), cutoffs)
    return {default_arm: (default, test_out, None), tuned_arm: (tuned, test_out, cutoffs)}

## 6. Training (same loop as Stage 3)

Per epoch: one pass over the training set, then predict the validation set. The **best epoch is chosen on validation QWK with the model's default rule** (arg-max for `ce`, rounding for `mse`), fixed before running. We keep validation *and* test outputs for the best epoch and for the 10 snapshot epochs, so that cut-offs can later be tuned on validation without retraining.

**Safety net:** after every epoch the log goes to `runs/<name>/log.csv` and a resumable checkpoint `last.pt` is written atomically. Re-running the notebook skips finished runs and resumes an interrupted one.

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def make_criterion(cfg):
    if cfg.loss == "ce":
        return nn.CrossEntropyLoss()
    if cfg.loss == "soft":                          # Gaussian soft labels: target mass falls off with distance from the true grade
        grades = torch.arange(NUM_CLASSES, device=device).float()
        def soft_ce(out, y):
            target = torch.softmax(-(grades.view(1, -1) - y.float().view(-1, 1)) ** 2 / (2 * cfg.sigma ** 2), dim=1)
            return -(target * F.log_softmax(out, dim=1)).sum(1).mean()
        return soft_ce
    if cfg.loss == "mse":                           # ordinal regression: squared distance to the true grade
        return lambda out, y: F.mse_loss(out[:, 0], y.float())
    raise ValueError(f"unknown loss {cfg.loss!r}")


def default_grades(out):
    # The rule used to pick the best epoch: arg-max for ce, rounding for mse.
    return out.argmax(1) if out.shape[1] == NUM_CLASSES else apply_cutoffs(out[:, 0], ROUNDING)


@torch.no_grad()
def predict(model, indices, cfg, batch_size=32):
    # Softmax probabilities (n, 5) for ce and soft, the raw regression output (n, 1) for mse.
    model.eval()
    out = []
    for start in range(0, len(indices), batch_size):
        x = to_model_input(images[indices[start:start + batch_size]])
        with torch.autocast(device.type, dtype=torch.float16, enabled=cfg.use_amp):
            logits = model(x).float()
        out.append((logits if cfg.loss == "mse" else torch.softmax(logits, dim=1)).cpu())
    return torch.cat(out).numpy()


def train_one_epoch(model, optimizer, scaler, criterion, cfg, split, rng):
    model.train()
    order = rng.permutation(split["train"])
    n_batches = len(order) // cfg.batch_size                                   # drop the last partial batch
    if cfg.max_train_batches:
        n_batches = min(n_batches, cfg.max_train_batches)
    total_loss, seen = 0.0, 0
    progress = tqdm(range(n_batches), desc="  training", leave=False)
    for b in progress:
        idx = np.sort(order[b * cfg.batch_size:(b + 1) * cfg.batch_size])      # sorted = faster memory reads
        x = to_model_input(images[idx])
        if cfg.augment:
            x = augment_batch(x)
        y = torch.from_numpy(labels[idx]).to(device)

        with torch.autocast(device.type, dtype=torch.float16, enabled=cfg.use_amp):
            out = model(x)
        loss = criterion(out.float(), y)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                                             # so clipping sees real gradient sizes
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(y)
        seen += len(y)
        progress.set_postfix(loss=f"{total_loss / seen:.3f}")
    return total_loss / seen

In [ ]:
def save_atomically(path, obj):
    # Write to a temp file, then rename: a crash mid-save can never leave a broken file behind.
    tmp = path.with_suffix(".tmp")
    try:
        torch.save(obj, tmp)
        tmp.replace(path)
    except (RuntimeError, OSError) as err:
        tmp.unlink(missing_ok=True)
        print(f"  warning: could not save {path.name} ({err})")


def load_run(run_dir, cfg):
    saved = np.load(run_dir / "outputs.npz")
    return {"config": cfg, "log": pd.read_csv(run_dir / "log.csv"), **{k: saved[k] for k in saved.files}}


def run_experiment(name, cfg):
    # Train one configuration and save its validation/test outputs. Resumes if interrupted.
    run_dir = RUNS_DIR / name
    run_dir.mkdir(parents=True, exist_ok=True)
    config_file, done_file = run_dir / "config.json", run_dir / "results.json"
    ckpt_path, best_path = run_dir / "last.pt", run_dir / "best_val_weights.pt"

    saved_cfg = json.loads(config_file.read_text()) if config_file.exists() else None
    if saved_cfg is not None:
        saved_cfg = {**{f.name: f.default for f in fields(Config)}, **saved_cfg}
    if saved_cfg is not None and saved_cfg != asdict(cfg) and done_file.exists():
        # Never throw away a finished run because a setting differs (e.g. a CPU runtime turns AMP off).
        changed = {k: (saved_cfg.get(k), v) for k, v in asdict(cfg).items() if saved_cfg.get(k) != v}
        raise RuntimeError(f"{name} already finished with other settings {changed} (saved, now). "
                           f"Delete {run_dir} by hand if you really want to retrain it.")
    if saved_cfg != asdict(cfg):              # a new experiment, or an unfinished one whose settings changed
        ckpt_path.unlink(missing_ok=True)
        done_file.unlink(missing_ok=True)
        config_file.write_text(json.dumps(asdict(cfg), indent=2))
    if done_file.exists():                    # finished in an earlier session
        print(f"=== {name} === already finished, loading saved outputs")
        return load_run(run_dir, cfg)

    split = SPLITS[cfg.dedupe]
    print(f"\n=== {name} ===  loss={cfg.loss}  sigma={cfg.sigma}  seed={cfg.seed}  lr={cfg.lr}  epochs={cfg.epochs}")
    seed_everything(cfg.seed)
    model = DRNet(cfg.backbone, cfg.pretrained, n_outputs(cfg)).to(device).to(memory_format=torch.channels_last)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    criterion = make_criterion(cfg)
    rng = np.random.default_rng(cfg.seed)

    snapshot_epochs = {round(cfg.epochs * k / cfg.n_snapshots) for k in range(1, cfg.n_snapshots + 1)}
    progress = {"log": [], "snap_val": [], "snap_test": [], "best_val_qwk": -2.0, "best_epoch": 0,
                "best_val": None, "best_test": None}
    first_epoch = 1

    if ckpt_path.exists():                                                     # continue an interrupted run
        try:
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        except Exception as err:
            ckpt = None
            print(f"  could not read {ckpt_path.name} ({err}); starting from scratch")
        if ckpt is not None:
            model.load_state_dict(ckpt["model"])
            optimizer.load_state_dict(ckpt["optimizer"])
            scaler.load_state_dict(ckpt["scaler"])
            rng.bit_generator.state = ckpt["rng_numpy"]
            torch.set_rng_state(ckpt["rng_torch"])
            if ckpt["rng_cuda"] is not None:
                torch.cuda.set_rng_state_all(ckpt["rng_cuda"])
            progress, first_epoch = ckpt["progress"], ckpt["epoch"] + 1
            print(f"  RESUMED after epoch {ckpt['epoch']} (best so far: val QWK {progress['best_val_qwk']:.3f})")

    y_val = labels[split["val"]]
    started, epochs_done_now = time.time(), 0
    for epoch in range(first_epoch, cfg.epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, optimizer, scaler, criterion, cfg, split, rng)

        val_out = predict(model, split["val"], cfg)
        val_pred = default_grades(val_out)
        val_qwk, val_acc = qwk(y_val, val_pred), float(accuracy_score(y_val, val_pred))
        progress["log"].append({"epoch": epoch, "train_loss": train_loss, "val_qwk": val_qwk, "val_acc": val_acc})

        notes, test_out = "", None
        if val_qwk > progress["best_val_qwk"]:                                 # new best on validation
            test_out = predict(model, split["test"], cfg)
            progress.update(best_val_qwk=val_qwk, best_epoch=epoch, best_val=val_out, best_test=test_out)
            save_atomically(best_path, model.state_dict())
            notes += " NEW BEST"
        if epoch in snapshot_epochs:                                           # ensemble member
            progress["snap_val"].append(val_out)
            progress["snap_test"].append(test_out if test_out is not None else predict(model, split["test"], cfg))
            notes += f" snapshot {len(progress['snap_test'])}/{cfg.n_snapshots}"

        pd.DataFrame(progress["log"]).to_csv(run_dir / "log.csv", index=False)    # live log, readable at any time
        if epoch % CHECKPOINT_EVERY == 0 and epoch < cfg.epochs:
            save_atomically(ckpt_path, {
                "epoch": epoch, "progress": progress, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(), "rng_numpy": rng.bit_generator.state, "rng_torch": torch.get_rng_state(),
                "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None})

        epochs_done_now += 1
        eta_min = (time.time() - started) / epochs_done_now * (cfg.epochs - epoch) / 60
        print(f"[{name}] epoch {epoch:2d}/{cfg.epochs} | loss {train_loss:.3f} | val QWK {val_qwk:.3f} "
              f"(best {progress['best_val_qwk']:.3f} @ep{progress['best_epoch']}) | val acc {val_acc:.3f} | "
              f"{time.time() - t0:.0f}s | ~{eta_min:.0f} min left |{notes}")

    np.savez_compressed(run_dir / "outputs.npz", y_val=y_val, y_test=labels[split["test"]],
                        val_idx=split["val"], test_idx=split["test"],
                        best_val=progress["best_val"], best_test=progress["best_test"],
                        snap_val=np.stack(progress["snap_val"]), snap_test=np.stack(progress["snap_test"]))
    done_file.write_text(json.dumps({"best_epoch": progress["best_epoch"],              # written last = run complete
                                     "best_val_qwk": progress["best_val_qwk"]}, indent=2))
    ckpt_path.unlink(missing_ok=True)
    return load_run(run_dir, cfg)

## 7. Run the experiments

Order: `soft_s42`, `soft_s43`, `soft_s44`, then the sigma = 1.0 ablation. Each run saves the validation and test outputs of the best epoch and of the 10 snapshots. Nothing from the test set is used to make a decision.

In [ ]:
runs = {}
for name, cfg in EXPERIMENTS.items():
    runs[name] = run_experiment(name, cfg)

## 8. Baselines from the earlier stages

The project repository holds the Stage 4 / 4e results. We read A, B and D per seed from `experiments/06_imbalance_losses/results/results_per_seed.csv` (group `main`, Colab runs), and re-score the two seed-42 Kaggle control runs from their saved outputs, because those give the per-image predictions that a paired bootstrap needs.

In [ ]:
REPO_URL = "https://github.com/devxsubh/aptos-dr-grading-reproduction"
BASELINE_REL = Path("experiments/06_imbalance_losses/results")        # arms A, B, D: Stage 4 / 4e outputs
REPO_DIR = LOCAL_DIR / "aptos-dr-repo"
if (BASELINE_REL / "runs").exists():                                  # notebook started from the repository root
    RES_DIR = BASELINE_REL
elif (Path("../..") / BASELINE_REL / "runs").exists():                # started from inside an experiment folder
    RES_DIR = Path("../..") / BASELINE_REL
else:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    RES_DIR = REPO_DIR / BASELINE_REL

baseline = pd.read_csv(RES_DIR / "results_per_seed.csv")
baseline_main = baseline[baseline["group"] == "main"]
print("baseline arms available:", sorted(baseline_main["arm"].unique()), "| seeds:", sorted(baseline_main["seed"].unique()))

def load_control(name, loss):
    saved = np.load(RES_DIR / "runs" / name / "outputs.npz")
    return {"config": replace(BASE_CFG, loss=loss, seed=42), **{k: saved[k] for k in saved.files}}

controls = {"ce": load_control("ce_s42_k", "ce"), "mse": load_control("mse_s42_k", "mse")}

## 9. Results: every arm, every seed

One row per (seed, prediction mode, arm). *single* = the best-validation epoch, *ensemble* = the 10 snapshots. Arms A, B, D are the Stage 4 numbers (Colab); G, H are the new soft-label runs.

In [ ]:
rows, decisions, cutoff_log = [], {}, {}
for name, run in runs.items():
    cfg = run["config"]
    for mode in ("single", "ensemble"):
        for arm, (pred, out_, cutoffs) in decide(run, mode).items():
            y = run["y_test"]
            decisions[(name, mode, arm)] = (y, pred, out_)
            if cutoffs is not None:
                cutoff_log[f"{name} {mode} {arm}"] = [round(float(c), 2) for c in cutoffs]
            m = compute_metrics(y, pred, out_)
            rows.append({"run": name, "sigma": cfg.sigma, "seed": cfg.seed, "mode": mode, "arm": arm,
                         **{k: v for k, v in m.items() if k != "confusion_matrix"}})
new = pd.DataFrame(rows)
new.to_csv(STAGE5_DIR / "results_per_seed_new.csv", index=False)
(STAGE5_DIR / "cutoffs.json").write_text(json.dumps(cutoff_log, indent=2))

SHOW = ["qwk", "accuracy", "macro_f1", "recall_Mild", "recall_Severe", "recall_PDR", "off_by_2_or_more",
        "ref_sensitivity", "ref_specificity", "ref_auc"]
main5 = new[new["sigma"] == 0.5]
def mean_std(df):
    agg = df.groupby(["mode", "arm"])[SHOW].agg(["mean", "std"])
    return pd.DataFrame({c: agg[(c, "mean")].map("{:.3f}".format) + " ± " + agg[(c, "std")].fillna(0).map("{:.3f}".format) for c in SHOW})
old = baseline_main[baseline_main["arm"].isin(["A", "B", "D"])]
table = pd.concat([mean_std(old), mean_std(main5)]).sort_index()
table.to_csv(STAGE5_DIR / "results_mean_over_seeds.csv")
print(f"Seeds finished (sigma 0.5): {sorted(main5['seed'].unique())}.  A = CE vote, B = CE + tuned cut-offs, D = ordinal MSE + tuned, "
      "G = soft labels + vote, H = soft labels + tuned cut-offs (hypothesis)\n")
display(table)
display(new.style.format(precision=3).hide(axis="index"))

### Paired bootstrap: is a difference real or test-set luck?

On the same 550 test images (seed 42, so the Kaggle control runs of arms A, B, D can be compared with the new runs one image at a time). Resample the images 2,000 times and compute the QWK difference each time. If the 95% interval excludes 0, the difference is not just which images ended up in the test set. It does **not** cover training randomness; the 3 seeds do that.

In [ ]:
def paired_bootstrap(y, pred_a, pred_b, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[b] = qwk(y[idx], pred_b[idx]) - qwk(y[idx], pred_a[idx])
    return qwk(y, pred_b) - qwk(y, pred_a), *np.percentile(diffs, [2.5, 97.5])

ctrl = {"ce": decide(controls["ce"], "ensemble"), "mse": decide(controls["mse"], "ensemble")}
y0 = controls["ce"]["y_test"]
assert np.array_equal(y0, runs["soft_s42"]["y_test"]), "test sets differ: the baselines are not comparable"
ctrl_pred = {"A": ctrl["ce"]["A"][0], "B": ctrl["ce"]["B"][0], "D": ctrl["mse"]["D"][0]}
soft_pred = {"G": decisions[("soft_s42", "ensemble", "G")][1], "H": decisions[("soft_s42", "ensemble", "H")][1]}

contrasts = [("G - A", "A", "G"), ("H - B", "B", "H"), ("H - D", "D", "H"), ("H - A", "A", "H")]
if "soft_sigma1_s42" in runs:
    soft_pred["H1"] = decisions[("soft_sigma1_s42", "ensemble", "H")][1]
    contrasts.append(("H(sigma 1.0) - H(sigma 0.5)", "H", "H1"))
boot_rows = []
for label, a, b in contrasts:
    pa = ctrl_pred[a] if a in ctrl_pred else soft_pred[a]
    pb = soft_pred[b]
    delta, lo, hi = paired_bootstrap(y0, pa, pb)
    boot_rows.append({"contrast": label, "delta_qwk": delta, "ci_low": lo, "ci_high": hi, "excludes_0": lo > 0 or hi < 0})
bootstrap = pd.DataFrame(boot_rows)
bootstrap.to_csv(STAGE5_DIR / "paired_bootstrap.csv", index=False)
display(bootstrap.style.format(precision=3).hide(axis="index"))

## 10. Were the predictions right? (M4)

The thresholds below are copied from [`experiments/07_soft_labels/hypothesis.md`](hypothesis.md), section 4, and were fixed before the first run. Every prediction is reported as it falls, including failures.

In [ ]:
PREDICTIONS = {      # copied from experiments/07_soft_labels/hypothesis.md, section 4
    "Q1_delta_qwk_range": (0.005, 0.025),   # mean QWK of G - A inside this range
    "Q2_min_relative_drop_off2": 0.20,      # errors >= 2 grades of H at least 20% below A
    "Q3_max_abs_h_minus_d": 0.01,           # abs(H - D) < 0.01
    "Q4_min_severe_recall_g": 0.23,         # Severe recall of G >= 0.23
    "Q5_max_abs_ref_auc_change": 0.01,      # referable AUC of G vs A
    "Q7_max_sigma1_gain": 0.005,            # sigma 1.0 does not beat sigma 0.5 by more than this
    "Q8_max_control_gap": 0.015,            # control re-score vs stored Stage 4e value
}
num = lambda s: float(str(s).split()[0])
ens = table.xs("ensemble", level="mode")
v = lambda arm, col: num(ens.loc[arm, col])

checks = []
lo, hi = PREDICTIONS["Q1_delta_qwk_range"]
d = v("G", "qwk") - v("A", "qwk")
checks.append(("Q1", "G - A QWK between +0.005 and +0.025", f"G - A = {d:+.3f}", lo <= d <= hi))
checks.append(("Q2", "errors >= 2 grades of H at least 20% below A",
               f"{v('A', 'off_by_2_or_more'):.1f} -> {v('H', 'off_by_2_or_more'):.1f}",
               v("H", "off_by_2_or_more") <= (1 - PREDICTIONS["Q2_min_relative_drop_off2"]) * v("A", "off_by_2_or_more")))
checks.append(("Q3", "H ties the Stage 4 ordinal regression (abs(H - D) < 0.01)",
               f"H {v('H', 'qwk'):.3f} vs D {v('D', 'qwk'):.3f}", abs(v("H", "qwk") - v("D", "qwk")) < PREDICTIONS["Q3_max_abs_h_minus_d"]))
checks.append(("Q4", "Severe recall of G stays >= 0.23",
               f"A {v('A', 'recall_Severe'):.3f} -> G {v('G', 'recall_Severe'):.3f}", v("G", "recall_Severe") >= PREDICTIONS["Q4_min_severe_recall_g"]))
checks.append(("Q5", "referable AUC of G within 0.01 of A",
               f"A {v('A', 'ref_auc'):.3f} -> G {v('G', 'ref_auc'):.3f}", abs(v("G", "ref_auc") - v("A", "ref_auc")) <= PREDICTIONS["Q5_max_abs_ref_auc_change"]))
checks.append(("Q6", "tuned cut-offs add more than the soft target: (H - G) > (G - A)",
               f"H - G {v('H', 'qwk') - v('G', 'qwk'):+.3f} vs G - A {v('G', 'qwk') - v('A', 'qwk'):+.3f}",
               (v("H", "qwk") - v("G", "qwk")) > (v("G", "qwk") - v("A", "qwk"))))
if "soft_sigma1_s42" in runs:
    q05 = float(new[(new.run == "soft_s42") & (new["mode"] == "ensemble") & (new.arm == "H")]["qwk"].iloc[0])
    q10 = float(new[(new.run == "soft_sigma1_s42") & (new["mode"] == "ensemble") & (new.arm == "H")]["qwk"].iloc[0])
    checks.append(("Q7", "sigma 1.0 does not beat sigma 0.5 by more than 0.005 (seed 42, rule H)",
                   f"sigma 0.5 {q05:.3f}, sigma 1.0 {q10:.3f}", q10 - q05 <= PREDICTIONS["Q7_max_sigma1_gain"]))
stored = baseline[(baseline["group"] == "control") & (baseline["seed"] == 42) & (baseline["mode"] == "ensemble")].set_index("arm")["qwk"]
gap = max(abs(qwk(y0, ctrl_pred["A"]) - stored["A"]), abs(qwk(y0, ctrl_pred["D"]) - stored["D"]))
checks.append(("Q8", "sanity: re-scored controls match Stage 4e (A and D, seed 42)", f"largest gap {gap:.4f}", gap <= PREDICTIONS["Q8_max_control_gap"]))

verdict = pd.DataFrame(checks, columns=["id", "prediction", "observed", "confirmed"])
verdict.to_csv(STAGE5_DIR / "prediction_check.csv", index=False)
print(f"{int(verdict['confirmed'].sum())} of {len(verdict)} predictions confirmed")
verdict.style.hide(axis="index")

## 11. Figures

**Validation curves** for each soft-label run, and **confusion matrices** (seed 42, 10-snapshot ensemble) for the baseline A, the Stage 4 ordinal D and the hypothesis H. Rows are the true grade, columns the predicted grade.

In [ ]:
fig, axes = plt.subplots(1, len(SEEDS), figsize=(5.2 * len(SEEDS), 3.8), squeeze=False)
for ax, seed in zip(axes[0], SEEDS):
    log = runs[f"soft_s{seed}"]["log"]
    ax.plot(log["epoch"], log["val_qwk"], label="val QWK")
    ax.plot(log["epoch"], log["train_loss"], label="train loss (soft CE)")
    ax.set_title(f"soft labels, seed {seed}"); ax.set_xlabel("epoch"); ax.set_ylim(0, 1); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "curves.png", dpi=150); plt.show()

panels = [("A: cross-entropy, vote", y0, ctrl_pred["A"]), ("D: ordinal MSE, tuned", y0, ctrl_pred["D"]),
          ("H: soft labels, tuned", y0, soft_pred["H"])]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (title, y, pred) in zip(axes, panels):
    cm = confusion_matrix(y, pred, labels=list(range(NUM_CLASSES)))
    ax.imshow(cm, cmap="Blues")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30); ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_title(f"{title}\nQWK {qwk(y, pred):.3f}"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.tight_layout(); fig.savefig(FIG_DIR / "confusion_matrices.png", dpi=150); plt.show()

## 12. Failure analysis (M5)

Three questions about the errors of the hypothesis arm H (seed 42, ensemble): which grades get confused, do the rare grades improve, and what do the misgraded images look like?

In [ ]:
recall = lambda y, pred: pd.Series([(pred[y == c] == c).mean() for c in range(NUM_CLASSES)], index=CLASS_NAMES)
per_class = pd.DataFrame({"A (CE vote)": recall(y0, ctrl_pred["A"]), "D (ordinal MSE)": recall(y0, ctrl_pred["D"]),
                          "G (soft, vote)": recall(y0, soft_pred["G"]), "H (soft, tuned)": recall(y0, soft_pred["H"])})
per_class["test images"] = np.bincount(y0, minlength=NUM_CLASSES)
per_class.to_csv(STAGE5_DIR / "per_class_recall.csv")
display(per_class.style.format(precision=3))

err = np.abs(y0 - soft_pred["H"])
print(f"H: {int((err == 0).sum())} correct, {int((err == 1).sum())} one grade off, {int((err >= 2).sum())} two or more grades off")

# misgraded images of arm H, worst first: show up to 8
test_idx = runs["soft_s42"]["test_idx"]
worst = np.argsort(-err)[: int((err > 0).sum())][:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8.4))
for ax, k in zip(axes.ravel(), worst):
    ax.imshow(preprocess(df["path"].iloc[test_idx[k]], 300))
    ax.set_title(f"true {CLASS_NAMES[y0[k]]} -> pred {CLASS_NAMES[soft_pred['H'][k]]}", fontsize=10); ax.axis("off")
fig.tight_layout(); fig.savefig(FIG_DIR / "failure_images.png", dpi=130); plt.show()

# are misgraded images blurrier? (same check as Stage 4 section 12)
from scipy.stats import mannwhitneyu
def sharpness(path):
    g = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(g, cv2.CV_64F).var())
sharp = np.array([sharpness(df["path"].iloc[i]) for i in test_idx])
wrong = err > 0
stat = mannwhitneyu(sharp[wrong], sharp[~wrong])
print(f"Laplacian sharpness: misgraded median {np.median(sharp[wrong]):.1f}, correct median {np.median(sharp[~wrong]):.1f}, p = {stat.pvalue:.4f}")

## 13. Writing it up (M5)

**Which numbers go in the paper**

* **Main table:** the mean ± std over 3 seeds for A, B, D, G, H (section 9), plus the paired bootstrap intervals (section 9) and the prediction check (section 10). Stage 3/4/4e rows stay in the table as the record of earlier experiments.
* **Failure analysis:** the confusion matrices, per-class recall and misgraded images (sections 11 and 12).
* **Reproduction gap:** unchanged from Stage 3/4 (paper 0.954 vs ours about 0.90–0.92).
* **Limitations:** one dataset, small test set (29 Severe, 44 PDR), no external validation, no clinical claim.

**Files written to Drive** under `MyDrive/dr-grading/stage5/`: `results_per_seed_new.csv`, `results_mean_over_seeds.csv`, `paired_bootstrap.csv`, `prediction_check.csv`, `per_class_recall.csv`, `cutoffs.json`, `figures/`, and `runs/<name>/` for each run. Bring these back into `experiments/07_soft_labels/results/` and fill in `experiments/07_soft_labels/results.md`.